# A checked RNA-seq analysis input

This notebook exports the compact `rnaseq-005` case and inspects count quality and its paired design. The dataset is a published prostate tumour versus adjacent non-tumour comparison. The adjacent tissue is not a healthy control. Install `openomicsbench` before running the notebook.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from omicsbench import Collection

collection = Collection()
with TemporaryDirectory() as temporary:
    pack = Path(temporary) / 'inputs'
    collection.pack_rnaseq(pack, ['rnaseq-005'])
    case = pack / 'rnaseq-005'
    qc = collection.qc_rnaseq(case / 'counts.tsv', case / 'samples.tsv',
                              case / 'contrasts.tsv', 'rnaseq-005')
    print('Genes:', qc['features'], 'Samples:', qc['samples'])
    print('Design:', qc['design_check']['summary']['status'])
    print('Count QC:', qc['summary']['status'])
    for sample in qc['sample_qc'][:3]:
        print(sample['sample'], sample['library_size'], sample['detected_features'])
    assert qc['features'] == 8000 and qc['samples'] == 28
    assert qc['summary']['status'] == 'pass'


The check reports structural consistency and descriptive count metrics. It does not perform differential expression or infer sequencing quality from raw reads. When your own pipeline creates a DESeq2, edgeR or limma table, run `omicsbench rnaseq assess-de rnaseq-005 results.csv` to inspect adjusted-p calls and compare effect directions with the archived DESeq2 reference. Keep the original target-minus-reference contrast direction.